# Build dollar-exposure panel

Minimal date x ISIN panel of German sovereign bonds for the US-auction-shock LP.
Output: `data/build/datasets/dollar_panel.csv` with exactly four columns:
`date, isin, yield, dollar_measure`.

`dollar_measure` = lagged 5-day rolling mean of the share of the bond's outstanding
that is repo-financed (bank = cash borrower, i.e. long the bond) by banks, where each
bank's volume is weighted by its dollar-repo activity (its USD share of total repo
volume over the sample). High value = bond heavily held by banks that are very
active in dollar repo.

Notes:
- Set `BANK_SECTOR` to the bank code used in `mbf_sector_enrichment_20210531`
  (analogous to `'IF'` for investment funds in the old project).
- Adjust `REPO` and `YIELDS_CSV` to your local paths. The yields file is assumed to
  have the same layout as `bond_timeseries_v2.csv` (Dates, ISIN, YLD_YTM_ASK,
  YLD_YTM_BID, amt_issued, bond_maturity, ...).


In [ ]:
import pyodbc
import pandas as pd
import numpy as np

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()

In [ ]:
REPO = r'C:\Users\hermesf\Projects\DMO_spillover'
YIELDS_CSV = REPO + r'\data\input\bond_timeseries_v2.csv'
OUT_CSV = REPO + r'\data\build\datasets\dollar_panel.csv'

START = '2021-01-04'
END = '2025-04-30'          # end of the auction-shock sample
BANK_SECTOR = 'Bank'        # <-- set to the bank code in mbf_sector_enrichment_20210531

## 1. Yields (local csv), German sovereigns only

In [ ]:
df = pd.read_csv(YIELDS_CSV)
df['Dates'] = pd.to_datetime(df['Dates'])
df['bond_maturity'] = pd.to_datetime(df['bond_maturity'])
df = df[((df['bond_maturity'] - df['Dates']).dt.days / 365) >= 0]
df = df[df['ISIN'].str[:2] == 'DE']
df = df[(df['Dates'] >= START) & (df['Dates'] <= END)]

In [ ]:
df['amt_issued'] = (
    df['amt_issued']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.replace(' ', '', regex=False)
)
df['amt_issued'] = pd.to_numeric(df['amt_issued'], errors='coerce')
df['amt_issued'] = df['amt_issued']/1e9
df['amt_issued'] = df['amt_issued'].fillna(df['amt_issued'].median())

In [ ]:
# mid yield, keep only rows where it exists
df['yield'] = (df['YLD_YTM_BID'] + df['YLD_YTM_ASK']) / 2
df = df[df['yield'].isna() == False]

securities = tuple(df['ISIN'].unique())

## 2. Bank-level dollar-repo activity (USD share of total repo volume, all collateral/ccy/clearing)

In [ ]:
# borrower side
query = f"""

SELECT borrower_id as bank_id,
sum(nominal_euro) as total_vol,
sum(CASE WHEN nominal_ccy = 'USD' THEN nominal_euro ELSE 0 END) as usd_vol

FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
WHERE s.business_date >= '{START}' AND s.business_date <= '{END}'
AND s_borrower.sector = '{BANK_SECTOR}'
GROUP BY borrower_id
"""

usd_b = pd.read_sql_query(query, cnxn)

In [ ]:
# lender side
query = f"""

SELECT lender_id as bank_id,
sum(nominal_euro) as total_vol,
sum(CASE WHEN nominal_ccy = 'USD' THEN nominal_euro ELSE 0 END) as usd_vol

FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date >= '{START}' AND s.business_date <= '{END}'
AND s_lender.sector = '{BANK_SECTOR}'
GROUP BY lender_id
"""

usd_l = pd.read_sql_query(query, cnxn)

In [ ]:
banks = (
    pd.concat([usd_b, usd_l])
      .groupby('bank_id', as_index=False)[['total_vol', 'usd_vol']].sum()
)
banks['usd_share'] = banks['usd_vol'] / banks['total_vol']

print(len(banks), 'banks')
banks['usd_share'].describe()

## 3. Bank positions in German bonds (bank = borrower -> long the bond)

In [ ]:
query = f"""

SELECT s.business_date,
security_isin as isin,
borrower_id as bank_id,
sum(nominal_euro) as long_vol

FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
WHERE s.business_date >= '{START}' AND s.business_date <= '{END}'
AND nominal_ccy IN ('EUR')
AND s_borrower.sector = '{BANK_SECTOR}'
AND gnlcoll = 'SPEC'
AND security_isin IN {securities}
GROUP BY business_date, isin, borrower_id
ORDER BY business_date, isin

"""

df_bank = pd.read_sql_query(query, cnxn)
df_bank['business_date'] = pd.to_datetime(df_bank['business_date'])

## 4. Bond-day dollar exposure -> `dollar_measure`

In [ ]:
# weight each bank's financing volume by its dollar-repo activity
df_bank = df_bank.merge(banks[['bank_id', 'usd_share']], on='bank_id', how='left')
df_bank['w_vol'] = df_bank['long_vol'] * df_bank['usd_share']

bond_day = df_bank.groupby(['business_date', 'isin'], as_index=False)['w_vol'].sum()

In [ ]:
out = df.merge(bond_day, left_on=['Dates', 'ISIN'], right_on=['business_date', 'isin'], how='left')
out.drop(columns=['business_date', 'isin'], inplace=True)
out['w_vol'] = out['w_vol'].fillna(0)

# % of outstanding financed by dollar-active banks
out['dollar_exposure'] = (out['w_vol']/1e9 / out['amt_issued']) * 100

In [ ]:
# predetermined version: lagged 5-day rolling mean (same convention as hf_intensity_pre)
out = out.sort_values(['ISIN', 'Dates'])

roll = (
    out.groupby('ISIN')['dollar_exposure']
     .apply(lambda s: s.rolling(window=5, min_periods=3).mean().shift(1))
     .reset_index(level=0, drop=True)
)

out['dollar_measure'] = roll.fillna(0)

## 5. Save

In [ ]:
panel = out.rename(columns={'Dates': 'date', 'ISIN': 'isin'})[['date', 'isin', 'yield', 'dollar_measure']]
panel.to_csv(OUT_CSV, index=False)

print(panel.shape)
panel['dollar_measure'].describe()